# PMM Dynamic Screener — MEXC Public REST

This notebook screens **MEXC** markets for **PMM Dynamic (Predictive Market Making)** using only public market-data endpoints. It is a **pre-ingestion gate**: the output is a ranked shortlist plus a candle-ingestor manifest, selected `BASE-QUOTE` pairs, and rule-estimate metadata.

This notebook is **public-data only**. It uses unauthenticated MEXC spot market endpoints and does not require or send any account credentials.

Stop-ship stance: a pair passing this notebook is **fit for research and ingestion only** until it survives candle-quality checks, walk-forward validation, and live microstructure review.


In [1]:
import os
import sys
import subprocess
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

CANDIDATE_DIRS = [
    Path("/quants-lab/research_notebooks/market_lab/pmm_dynamic"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
PMM_DIR = next((p.resolve() for p in CANDIDATE_DIRS if (p / "pyproject.toml").exists() and (p / "pmm_lab").exists()), None)
if PMM_DIR is None:
    raise FileNotFoundError("Could not locate the pmm_dynamic project root. Open this notebook from inside the pmm_dynamic repo.")

if str(PMM_DIR) not in sys.path:
    sys.path.insert(0, str(PMM_DIR))
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(PMM_DIR), "--quiet"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"PMM project root: {PMM_DIR}")


from pmm_lab.screener import MEXCPublicScreener, default_mexc_config, export_screening_artifacts
from pmm_lab.screener.common import compute_coarse_scores, select_shortlist
from pmm_lab.screener.mexc_public import MEXC_BASE_URL


PMM project root: /quants-lab/research_notebooks/market_lab/pmm_dynamic


## 1. Configuration

Edit the universe, thresholds, and output path here. Keep the defaults conservative unless you explicitly want a wider exploratory net.


In [2]:
# ============================================================
# USER CONFIG — edit these, then Run All
# ============================================================
QUOTE_ASSET = "*"
INTERVAL = "5m"
UNIVERSE_TOP_K = 100
FINAL_TOP_N = 30
CANDLE_LIMIT = 288
DEPTH_LIMIT = 200
RECENT_TRADE_LIMIT = 500

# Optional pair overrides.
# Keep INCLUDE_SYMBOLS empty to use the full eligible universe.
INCLUDE_SYMBOLS = []
EXCLUDE_SYMBOLS = []

OUTPUT_ROOT = (
    PMM_DIR / "artifacts" / "screener" / "mexc" /
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

cfg = default_mexc_config()
cfg.quote_asset = QUOTE_ASSET
cfg.interval = INTERVAL
cfg.universe_top_k = UNIVERSE_TOP_K
cfg.final_top_n = FINAL_TOP_N
cfg.candle_limit = CANDLE_LIMIT
cfg.depth_limit = DEPTH_LIMIT
cfg.recent_trade_limit = RECENT_TRADE_LIMIT
cfg.include_symbols = tuple(INCLUDE_SYMBOLS)
cfg.exclude_symbols = tuple(EXCLUDE_SYMBOLS)

# Conservative defaults. Loosen only if you explicitly want more exploratory coverage.
cfg.min_quote_volume_24h = 1000000.0
cfg.max_spread_bps = 50.0
cfg.min_top_of_book_quote = 250.0
cfg.min_depth_10bps_quote = 1000.0
cfg.max_last_trade_age_sec = 1800.0
cfg.min_recent_trade_count = 100
cfg.min_candle_count = 240
cfg.min_candle_coverage_ratio = 0.97
cfg.max_zero_volume_fraction = 0.2
cfg.min_natr_bps = 10.0
cfg.max_natr_bps = 250.0

print("Screener config")
display(pd.Series(cfg.as_dict()).to_frame("value"))
print(f"Output root: {OUTPUT_ROOT}")


Screener config


,value
connector,mexc
quote_asset,*
interval,5m
universe_top_k,100
final_top_n,30
candle_limit,288
depth_limit,200
recent_trade_limit,500
request_pause_sec,0.12
timeout_seconds,30.0


Output root: /quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949


## 2. Build the cheap universe snapshot

This step uses only batch endpoints to discover the broad market universe and assign a coarse score before any per-pair enrichment calls are made.


In [3]:
screener = MEXCPublicScreener(cfg)

universe = screener.build_universe()
universe = compute_coarse_scores(universe)
shortlist = select_shortlist(universe, cfg)

summary = pd.DataFrame([
    {
        "connector": cfg.connector,
        "quote_asset": cfg.quote_asset,
        "interval": cfg.interval,
        "universe_rows": len(universe),
        "shortlist_rows": len(shortlist),
    }
])
display(summary)

preview_cols = [
    c for c in [
        "trading_pair",
        "exchange_symbol",
        "quote_volume_24h",
        "spread_bps",
        "coarse_score",
        "price_tick_estimate",
        "amount_step_estimate",
        "status_detail",
    ]
    if c in universe.columns
]

display(universe.loc[:, preview_cols].head(20))
print(f"Shortlist for detailed enrichment: {len(shortlist)}")
display(shortlist.loc[:, preview_cols].head(min(25, len(shortlist))))


,connector,quote_asset,interval,universe_rows,shortlist_rows
0,mexc,*,5m,2403,100


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,ETH-USDT,ETHUSDT,9.092971e+08,0.046590,99.904256,0.010000,0.000100,1
1,GOLD(PAXG)-USDT,GOLD(PAXG)USDT,5.719428e+07,0.022924,99.764827,0.010000,0.000001,1
2,USDC-USDT,USDCUSDT,3.251860e+07,0.999750,99.598328,0.000100,1.000000,1
3,USD1-USDT,USD1USDT,1.586873e+07,1.000350,99.349653,0.000100,0.000000,1
4,BNB-USDT,BNBUSDT,4.900773e+07,0.158074,99.314170,0.010000,0.001000,1
5,XRP-USDT,XRPUSDT,1.164552e+08,1.414527,99.132011,0.000100,0.100000,1
6,SUI-USDT,SUIUSDT,2.342215e+07,1.061740,98.926009,0.000100,0.000000,1
7,SOL-USDT,SOLUSDT,1.536872e+08,1.103205,98.903017,0.010000,0.010000,1
8,USDE-USDT,USDEUSDT,5.753759e+06,1.000450,98.859627,0.000100,0.000000,1
9,LTC-USDT,LTCUSDT,1.037177e+07,1.801315,98.699220,0.010000,0.001000,1


Shortlist for detailed enrichment: 100


,trading_pair,exchange_symbol,quote_volume_24h,spread_bps,coarse_score,price_tick_estimate,amount_step_estimate,status_detail
0,ETH-USDT,ETHUSDT,9.092971e+08,0.046590,99.904256,0.010000,0.000100,1
1,GOLD(PAXG)-USDT,GOLD(PAXG)USDT,5.719428e+07,0.022924,99.764827,0.010000,0.000001,1
2,USDC-USDT,USDCUSDT,3.251860e+07,0.999750,99.598328,0.000100,1.000000,1
3,USD1-USDT,USD1USDT,1.586873e+07,1.000350,99.349653,0.000100,0.000000,1
4,BNB-USDT,BNBUSDT,4.900773e+07,0.158074,99.314170,0.010000,0.001000,1
5,XRP-USDT,XRPUSDT,1.164552e+08,1.414527,99.132011,0.000100,0.100000,1
6,SUI-USDT,SUIUSDT,2.342215e+07,1.061740,98.926009,0.000100,0.000000,1
7,SOL-USDT,SOLUSDT,1.536872e+08,1.103205,98.903017,0.010000,0.010000,1
8,USDE-USDT,USDEUSDT,5.753759e+06,1.000450,98.859627,0.000100,0.000000,1
9,LTC-USDT,LTCUSDT,1.037177e+07,1.801315,98.699220,0.010000,0.001000,1


## 3. Run the detailed screener

This step enriches the shortlist with order-book depth, recent trades, and candles, then applies hard gates plus the final PMM-oriented score.


In [4]:
run = screener.screen_from_universe(universe)
final_df = run.final.copy()
selected_df = run.selected.copy()

status_counts = pd.DataFrame([
    {
        "enriched_rows": len(final_df),
        "selected_rows": len(selected_df),
        "pass_rate": float(selected_df.shape[0] / final_df.shape[0]) if len(final_df) else 0.0,
    }
])
display(status_counts)

final_cols = [
    c for c in [
        "trading_pair",
        "screen_score",
        "passed_filters",
        "quote_volume_24h",
        "spread_bps",
        "top_of_book_quote",
        "sym_depth_quote_10bps",
        "recent_trade_count",
        "last_trade_age_sec",
        "n_candles",
        "coverage_ratio",
        "zero_volume_fraction",
        "natr_bps_mean",
        "efficiency_ratio",
        "rejection_reason",
    ]
    if c in final_df.columns
]
display(final_df.loc[:, final_cols].head(min(50, len(final_df))))


,enriched_rows,selected_rows,pass_rate
0,100,27,0.27


,trading_pair,screen_score,passed_filters,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,n_candles,coverage_ratio,zero_volume_fraction,natr_bps_mean,efficiency_ratio,rejection_reason
0,SOL-USDT,92.066385,True,1.536872e+08,1.103205,2.177685e+03,6.793480e+05,500,6.920546,288,1.0,0.000000,34.292613,0.099015,
1,DOGE-USDT,91.014450,True,3.997877e+07,1.068662,3.895813e+03,2.749801e+05,500,3.646674,288,1.0,0.000000,31.371570,0.065978,
2,ADA-USDT,85.096000,True,1.987415e+07,3.820439,7.053730e+03,8.049767e+04,500,2.066716,288,1.0,0.000000,34.580912,0.078493,
3,LTC-USDT,84.448000,True,1.037177e+07,1.801315,1.581946e+03,7.650489e+04,500,11.675224,288,1.0,0.000000,23.401964,0.083605,
4,BNB-USDT,84.096329,True,4.900773e+07,0.158072,5.729097e+02,9.182993e+04,500,5.326376,288,1.0,0.000000,20.936207,0.029595,
5,LINK-USDT,84.032458,True,3.241892e+07,1.096551,4.559500e+02,3.186715e+04,500,22.627482,288,1.0,0.000000,32.940267,0.091439,
6,GOLD(PAXG)-USDT,83.210071,True,5.719428e+07,0.022919,1.391359e+03,8.700509e+04,500,13.716707,288,1.0,0.000000,41.653120,0.004631,
7,TRX-USDT,81.637632,True,1.082191e+07,3.230496,1.071396e+04,1.461079e+05,500,16.438863,288,1.0,0.000000,14.336747,0.014409,
8,TAO-USDT,80.978000,True,2.508577e+07,4.450661,2.845514e+03,1.033551e+04,500,5.414564,288,1.0,0.000000,70.375819,0.129422,
9,UNI-USDT,75.224980,True,6.555532e+06,2.788234,8.468907e+02,1.586584e+04,500,26.002542,288,1.0,0.000000,33.734122,0.043925,


## 4. Diagnostics

Inspect which pairs passed, which pairs failed, and why. Rejection counts are often more useful than raw rankings because they show whether you are liquidity-bound, spread-bound, or data-quality-bound.


In [5]:
if final_df.empty:
    print("No enriched rows returned.")
else:
    passed = final_df[final_df["passed_filters"]].copy()
    rejected = final_df[~final_df["passed_filters"]].copy()

    print(f"Passed: {{len(passed)}} | Rejected: {{len(rejected)}}")

    if not passed.empty:
        display(
            passed.loc[:, [
                c for c in [
                    "trading_pair",
                    "screen_score",
                    "quote_volume_24h",
                    "spread_bps",
                    "top_of_book_quote",
                    "sym_depth_quote_10bps",
                    "recent_trade_count",
                    "last_trade_age_sec",
                    "natr_bps_mean",
                    "efficiency_ratio",
                ] if c in passed.columns
            ]].head(cfg.final_top_n)
        )

    if not rejected.empty:
        reject_counts = (
            rejected["rejection_reason"]
            .fillna("")
            .str.split("; ")
            .explode()
            .loc[lambda s: s.ne("")]
            .value_counts()
            .rename_axis("rejection_reason")
            .to_frame("count")
        )
        display(reject_counts.head(20))


Passed: {len(passed)} | Rejected: {len(rejected)}


,trading_pair,screen_score,quote_volume_24h,spread_bps,top_of_book_quote,sym_depth_quote_10bps,recent_trade_count,last_trade_age_sec,natr_bps_mean,efficiency_ratio
0,SOL-USDT,92.066385,1.536872e+08,1.103205,2177.684950,679348.004640,500,6.920546,34.292613,0.099015
1,DOGE-USDT,91.014450,3.997877e+07,1.068662,3895.813150,274980.103197,500,3.646674,31.371570,0.065978
2,ADA-USDT,85.096000,1.987415e+07,3.820439,7053.729760,80497.669859,500,2.066716,34.580912,0.078493
3,LTC-USDT,84.448000,1.037177e+07,1.801315,1581.946184,76504.885688,500,11.675224,23.401964,0.083605
4,BNB-USDT,84.096329,4.900773e+07,0.158072,572.909728,91829.926281,500,5.326376,20.936207,0.029595
5,LINK-USDT,84.032458,3.241892e+07,1.096551,455.950000,31867.148570,500,22.627482,32.940267,0.091439
6,GOLD(PAXG)-USDT,83.210071,5.719428e+07,0.022919,1391.358526,87005.092372,500,13.716707,41.653120,0.004631
7,TRX-USDT,81.637632,1.082191e+07,3.230496,10713.958776,146107.910110,500,16.438863,14.336747,0.014409
8,TAO-USDT,80.978000,2.508577e+07,4.450661,2845.513720,10335.509260,500,5.414564,70.375819,0.129422
9,UNI-USDT,75.224980,6.555532e+06,2.788234,846.890700,15865.843510,500,26.002542,33.734122,0.043925


,count
rejection_reason,
top_of_book_quote<250,56
depth_10bps<1000,36
quote_volume_24h<1e+06,27
natr_bps_mean<10,8
zero_volume_fraction>0.20,2


## 5. Export artifacts

This writes CSV, JSON, a Markdown report, a candle-ingestor manifest, and an exchange-rules patch for the selected pairs.


In [6]:
artifact_paths = export_screening_artifacts(
    run,
    output_dir=str(OUTPUT_ROOT),
    base_url=MEXC_BASE_URL,
    trade_limit=cfg.recent_trade_limit,
)

print("Artifacts written")
display(pd.Series(asdict(artifact_paths)).to_frame("path"))


Artifacts written


,path
root_dir,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949
universe_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/universe.csv
shortlist_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/shortlist.csv
final_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/final_screen.csv
selected_csv,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/selected_pairs.csv
selected_pairs_txt,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/selected_pairs.txt
selected_pairs_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/selected_pairs.json
symbol_metadata_json,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/symbol_metadata.json
candle_ingestor_manifest_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/candle_ingestor_manife...
exchange_rules_patch_yaml,/quants-lab/research_notebooks/market_lab/pmm_dynamic/artifacts/screener/mexc/20260324_040949/exchange_rules_patch.yaml


## 6. Merge into candle ingestion

The manifest below follows the same `exchanges -> pairs -> intervals` structure already used by your candle-ingest and candle-gap-repair tooling, with slash-formatted exchange pairs and Hummingbot-normalized selected pair files alongside it.


In [7]:
from pathlib import Path

print("Selected Hummingbot pairs")
print("-" * 80)
print(Path(artifact_paths.selected_pairs_txt).read_text())

print("\nCandle ingestor manifest")
print("-" * 80)
print(Path(artifact_paths.candle_ingestor_manifest_yaml).read_text())

print("\nExchange rules patch (estimates only)")
print("-" * 80)
print(Path(artifact_paths.exchange_rules_patch_yaml).read_text())


Selected Hummingbot pairs
--------------------------------------------------------------------------------
SOL-USDT
DOGE-USDT
ADA-USDT
LTC-USDT
BNB-USDT
LINK-USDT
GOLD(PAXG)-USDT
TRX-USDT
TAO-USDT
UNI-USDT
TRUMP-USDT
HBAR-USDT
WLD-USDT
TON-USDT
DOT-USDT
WXT-USDT
ZRO-USDT
ASTER-USDT
SHIB-USDT
PUMP-USDT
SAHARA-USDT
ICP-USDT
OP-USDT
ETC-USDT
RENDER-USDT
CRV-USDT
ATOM-USDT


Candle ingestor manifest
--------------------------------------------------------------------------------
backfill_days: 180
request_delay: 0.5
overlap_candles: 2
include_open_candle: false
http:
  timeout_seconds: 30.0
  max_retries: 3
  retry_backoff: 1.8
  user_agent: pmm-lab-screener/0.1
exchanges:
  mexc:
    enabled: true
    base_url: https://api.mexc.com
    pairs:
    - SOL/USDT
    - DOGE/USDT
    - ADA/USDT
    - LTC/USDT
    - BNB/USDT
    - LINK/USDT
    - GOLD(PAXG)/USDT
    - TRX/USDT
    - TAO/USDT
    - UNI/USDT
    - TRUMP/USDT
    - HBAR/USDT
    - WLD/USDT
    - TON/USDT
    - DOT/USDT
    - WXT/USD